In [2]:
from models import NeuralNet
from data import get_dataloader, DataPreprocessor, DataSplitter
from utils import load_config
import pandas as pd
import wandb
import pyreadr
import lightning as pl
from lightning.pytorch.loggers import WandbLogger

/home/cjs/miniconda3/envs/idp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Loading & Preprocessing

In [4]:
r_data = pyreadr.read_r("../../data/raw/features_all_windows_combined.rds")
df = pd.DataFrame(next(iter(r_data.values())))
config = load_config("../../config.yaml")

In [6]:
train_df, val_df, test_df = DataSplitter(test_size=0.15, val_size=0.15, random_state=42).group_split(df, config["dataset"].get("groupsplit_column", "user_id"))
preprocessor = DataPreprocessor(config)
train_df = preprocessor.fit_transform(train_df)
val_df = preprocessor.transform(val_df)
test_df = preprocessor.transform(test_df)

train_dataloader = get_dataloader(train_df, config, batch_size=32, shuffle=False, num_workers=4)
val_dataloader = get_dataloader(val_df, config, batch_size=32, shuffle=False, num_workers=4)
test_dataloader = get_dataloader(val_df, config, batch_size=32, shuffle=False, num_workers=4)

ValueError: A given column is not a column of the dataframe

## Model Training

In [ ]:
wandb_logger = WandbLogger(project=config.get("project", "situation-prediction"), name="nn_training")
model = NeuralNet(config, wandb_logger=wandb_logger)
trainer = pl.Trainer(max_epochs=config.get("max_epochs", 100),
                     logger=[wandb_logger],
                     precision=config.get("precision", "bf16"))
trainer.fit(model, 
            train_dataloader=train_dataloader,
            val_dataloader=val_dataloader)